# conv-stride-downsample — worked example 2: Strided conv equals dense conv then slice every S

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-stride-downsample`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A stride-`S` convolution computes exactly the same dot products as a stride-1 convolution but keeps only the outputs at positions `0, S, 2S, ...`. So a strided conv is equivalent to a dense (stride-1) conv followed by slicing `[..., ::S]`. This is why striding downsamples: it subsamples the dense response map.

## Worked solution

**Goal.** Show numerically that `F.conv1d(x, w, stride=S)` equals `F.conv1d(x, w, stride=1)[..., ::S]`.

**Step 1 — compute the dense response.** A stride-1 conv slides the kernel one step at a time, producing an output of length `(L - K) // 1 + 1 = L - K + 1`. Every possible window position is represented.

**Step 2 — subsample.** Striding by `S` means we only emit windows whose start index is a multiple of `S`. Indexing the dense output with `[..., ::S]` keeps positions `0, S, 2S, ...`, which are exactly those windows. So the strided output is a pure subset of the dense output — no new arithmetic, just dropped columns.

**Step 3 — confirm the length matches the formula.** The dense length is `L - K + 1`; slicing every `S` gives `ceil((L - K + 1) / S)`, which equals `(L - K) // S + 1`. For `L=20, K=4, S=3`: dense length `17`, sliced length `ceil(17/3)=6`, and `(20-4)//3+1 = 16//3+1 = 5+1 = 6`. They agree.

**Step 4 — verify equality.** We run both convs with identical weights and assert the strided output is `allclose` to the sliced dense output.

In [ ]:
import torch.nn.functional as F

def strided_equals_sliced(x, w, s):
    dense = F.conv1d(x, w, stride=1)
    strided = F.conv1d(x, w, stride=s)
    return strided, dense[..., ::s]

t.manual_seed(0)
N, C_in, C_out, L, K, S = 1, 2, 3, 20, 4, 3
x = t.randn(N, C_in, L)
w = t.randn(C_out, C_in, K)
strided, sliced = strided_equals_sliced(x, w, S)
print('strided len:', strided.shape[-1], 'sliced len:', sliced.shape[-1])
print('formula len:', (L - K) // S + 1)
print('allclose:', t.allclose(strided, sliced, atol=1e-5))